In [7]:
import pennylane as qml
import numpy as np

# ==========================================
# ATTO 1: I DATI E I BINARI (Registri)
# ==========================================
# 1. Partiamo da una VERA matrice diagonale 4x4 (L'input del Prof)
# X_matrice = np.array([
#     [ 0.5,   0,   0,   0],
#     [  0, -0.2,   0,   0],
#     [  0,   0, 0.8,   0],
#     [  0,   0,   0,   0.1]
# ])

# X_matrice = np.array([
#     [ 0.7,   0,   0,   0],
#     [  0, -4,   0,   0],
#     [  0,   0, 8,   0],
#     [  0,   0,   0,   0.3]
# ])

# X_matrice = np.array([
#     [ -720,   0,   0,   0],
#     [  0, 56,   0,   0],
#     [  0,   0, 200,   0],
#     [  0,   0,   0,   170]
# ])

# X_matrice = np.array([
#     [ -720,   0],
#     [  0, 56]
# ])

X_matrice = np.array([
    [ -720,   0,   0],
    [  0, 56,   0],
    [  0,   0, 200],
])

print("1. La matrice X di partenza è:")
print(X_matrice)

# 2. Estraiamo la diagonale classica
x_classico_grezzo = np.diag(X_matrice)
N_originale = len(x_classico_grezzo)

# 3. IL CERVELLO DINAMICO (Calcolo qubit e Padding)
# Calcoliamo n: il numero di qubit necessari (logaritmo in base 2 arrotondato per eccesso)
# Es: N=2 -> n=1. N=3 -> n=2. N=4 -> n=2.
n = int(np.ceil(np.log2(N_originale)))

# Calcoliamo quanti slot fisici avrà la memoria quantistica (la potenza di 2 più vicina)
N_pad = 2**n

# Facciamo il padding: aggiungiamo zeri alla fine se N_originale è minore di N_pad
# (Fondamentale per le matrici 3x3, 5x5, ecc.)
x_classico = np.pad(x_classico_grezzo, (0, N_pad - N_originale))

# 4. Normalizzazione quantistica
norma_originale = np.linalg.norm(x_classico)
x = x_classico / norma_originale 

# 5. ASSEGNAZIONE DINAMICA DEI FILI
# 5. Assegniamo i fili. n=2 (perché abbiamo 4 dati, 2^2=4)
# Quindi servono n fili per address, n per data, 1 per l'ancilla.
# Generiamo le liste dei fili in base alla 'n' calcolata!
qubits_address = list(range(0, n))                   # Es: se n=1 -> [0]
qubits_dati = list(range(n, 2*n))                    # Es: se n=1 -> [1]
ancilla_b = [2*n]                                    # Es: se n=1 -> [2]

# Prepariamo un raggruppamento che ci servirà per S_0
qubits_da_B = qubits_dati + ancilla_b 

# ==========================================
# ATTO 2: ORACOLO E RIFLESSIONE
# ==========================================

def U_x(wires):
    """
    L'Oracolo: Inietta il vettore x nelle ampiezze dei qubit bersaglio.
    (Sostituisce tutta la vecchia matematica di Mottonen)
    """
     #qml.AmplitudeEmbedding(features=x, wires=wires, normalize=True)
    qml.MottonenStatePreparation(x, wires=wires)

def S_0():
    """
    La Riflessione: La matrice diagonale con un -1 in alto a sinistra.
    Inverte il segno SOLO se i qubit 'dati' e 'B' sono tutti a zero.
    Non tocca gli 'address' per non rovinare le coordinate!
    """
    stato_zero = np.zeros(len(qubits_da_B), dtype=int)
    qml.FlipSign(stato_zero, wires=qubits_da_B)

# ==========================================
# ATTO 3: W (dalla Figura 1)
# ==========================================

def W():
    """
    Allinea i dati x_k con le coordinate corrette della matrice diagonale.
    """
    # 1. U normale solo sui dati
    U_x(wires=qubits_dati)
    
    # 2. Hadamard sull'ancilla B
    qml.Hadamard(wires=ancilla_b)
    
    # 3. U inverso sui dati, controllato da B
    # qml.adjoint ribalta l'oracolo, qml.ctrl lo subordina a B
    qml.ctrl(qml.adjoint(U_x), control=ancilla_b)(wires=qubits_dati)
    
    # 4. Le porte Toffoli incrociate (da address a dati, controllate da B)
    # Servono a "saldare" il dato corretto all'indirizzo corretto
    for i in range(len(qubits_address)):
        # Toffoli prende: [controllo_1, controllo_2, bersaglio]
        qml.Toffoli(wires=[ancilla_b[0], qubits_address[i], qubits_dati[i]])
        
    # 5. Hadamard finale sull'ancilla B
    qml.Hadamard(wires=ancilla_b)

# ==========================================
# ATTO 4: G
# ==========================================

def G():
    # Costruisce l'operatore G = W S_0 W^\dagger Z_B.
    # Le operazioni si scrivono nell'ordine esatto in cui toccano i qubit.
    
    # 1. Z_B (Riflessione sull'ancilla)
    qml.PauliZ(wires=ancilla_b)
    
    # 2. W^\dagger (Smontaggio: PennyLane inverte tutto W in automatico)
    qml.adjoint(W)()
    
    # 3. S_0 (Riflessione sull'origine)
    S_0()
    
    # 4. W (Rimontaggio)
    W()

# ==========================================
# 5. STAMPA E VERIFICA DELLA MATRICE
# ==========================================

# 1. Creiamo una lista unica con tutti i 5 fili nell'ordine corretto
tutti_i_fili = qubits_address + qubits_dati + ancilla_b

# 1. Calcoliamo la matrice 32x32 del circuito G
matrice_gigante = qml.matrix(G, wire_order=tutti_i_fili)()

# 2. Calcoliamo gli autovalori complessi della matrice
autovalori = np.linalg.eigvals(matrice_gigante)

# 3. Estraiamo la parte reale (i coseni degli angoli di rotazione)
# (Il segno meno compensa la rotazione speculare di G)
parte_reale = -np.real(autovalori)

# 4. Togliamo i doppioni per estrarre il cuore dei dati
valori_unici = np.unique(np.round(parte_reale, 4))

# 5. Rimuoviamo gli "scarti geometrici" (1.0 e -1.0, che sono gli assi fissi)
dati_estratti = [float(v) for v in valori_unici if abs(abs(v) - 1.0) > 0.0001]

# 6. Rigonfiamo i dati usando la scala che avevamo salvato all'inizio
dati_ricostruiti = [v * norma_originale for v in dati_estratti]

dati_ordinati = []
# Scorriamo l'array originale (che ha l'ordine giusto)
for val_orig in x_classico:
    # Troviamo tra i dati ricostruiti quello che "assomiglia" di più all'originale
    val_esatto = min(dati_ricostruiti, key=lambda x: abs(x - val_orig))
    dati_ordinati.append(val_esatto)

# 7. RICOSTRUZIONE DELLA MATRICE DIAGONALE
# np.diag prende la nostra lista e la trasforma nella matrice finale
matrice_finale = np.diag(dati_ordinati)

# --- STAMPA ---
print("========================================")
print("I dati originali normalizzati in ingresso:")
print(np.round(x, 4))
print("========================================\n")
print("I numeri estratti dagli AUTOVALORI di G in uscita:")
print(dati_estratti)
print("\nI numeri estratti e ricostruiti dagli AUTOVALORI di G in uscita:")
print(np.round(dati_ordinati, 1))
print("\nE la MATRICE DIAGONALE Block-Encodata in G è:")
print(np.round(matrice_finale, 1))
print("========================================")
print("\nOperazione completata con successo!")

1. La matrice X di partenza è:
[[-720    0    0]
 [   0   56    0]
 [   0    0  200]]
I dati originali normalizzati in ingresso:
[-0.9608  0.0747  0.2669  0.    ]

I numeri estratti dagli AUTOVALORI di G in uscita:
[-0.9608, 0.0, 0.0747, 0.2669]

I numeri estratti e ricostruiti dagli AUTOVALORI di G in uscita:
[-720.   56.  200.    0.]

E la MATRICE DIAGONALE Block-Encodata in G è:
[[-720.    0.    0.    0.]
 [   0.   56.    0.    0.]
 [   0.    0.  200.    0.]
 [   0.    0.    0.    0.]]

Operazione completata con successo!
